# 3. Fine-tune SmolVLA

Fine-tune `lerobot/smolvla_base` on the local SO-101 demonstrations. Run this on a CUDA-equipped training machine with the environment from notebook 1. Copy the complete dataset directory from the collection machine first if they differ. The pretrained model is downloaded from the Hugging Face Hub; check its access terms and your network connection. No dataset or model upload is enabled by these commands.

The example follows the [SmolVLA guide](https://huggingface.co/docs/lerobot/en/smolvla). Twenty thousand steps can take hours; GPU memory, dataset decoding and throughput depend on the machine. Begin with a short smoke test and adjust batch size for available memory.

In [ ]:
from pathlib import Path
import torch

dataset_root = Path.home() / 'lerobot-workshop-data' / 'so101-pick-place'
print('Dataset metadata:', (dataset_root / 'meta' / 'info.json').is_file())
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

If the dataset path differs, update both the check and the terminal variable. If CUDA is unavailable, the command below will not work as written; use a suitable GPU machine or consult the official guide for alternatives. In an activated terminal, set:

```bash
export DATASET_ID=workshop/so101-pick-place
export DATASET_ROOT="$HOME/lerobot-workshop-data/so101-pick-place"
export OUTPUT_DIR="$HOME/lerobot-workshop-runs/smolvla"
export RENAME_MAP='{"observation.images.top": "observation.images.camera1", "observation.images.wrist": "observation.images.camera2"}'
```

The mapping matches the `top` and `wrist` dataset keys to the pretrained model's `camera1` and `camera2`. Use the identical mapping at deployment. If you changed the camera names, update both commands consistently.

## Train

For an initial check, use `--steps=100` and a separate output directory, then run the full training with the command below. LeRobot refuses to overwrite an existing output directory: choose a new `OUTPUT_DIR` for each run. Lower `--batch_size` if GPU memory is insufficient.

```bash
lerobot-train \
  --policy.path=lerobot/smolvla_base \
  --dataset.repo_id="$DATASET_ID" --dataset.root="$DATASET_ROOT" \
  --rename_map="$RENAME_MAP" \
  --policy.device=cuda --policy.push_to_hub=false \
  --batch_size=8 --steps=20000 --save_freq=5000 \
  --output_dir="$OUTPUT_DIR" --job_name=so101_smolvla
```

Check the loss and data-loading logs for errors; loss alone does not establish real-world success. The saved inference artifact is `"$OUTPUT_DIR/checkpoints/last/pretrained_model"` after a checkpoint has been written. It includes the policy and processors. Copy the whole `pretrained_model` directory to the deployment machine (resolve the `last` symlink when copying), not just the weights. Keep the training configuration, camera mapping and task instruction with it. For checkpoint resume options, see `lerobot-train --help` and the [training guide](https://huggingface.co/docs/lerobot/en/smolvla).

In [ ]:
from pathlib import Path

model = Path.home() / 'lerobot-workshop-runs' / 'smolvla' / 'checkpoints' / 'last' / 'pretrained_model'
print('Default model path:', model)
print('Model configuration exists:', (model / 'config.json').is_file())

Update the check if `OUTPUT_DIR` differs. Next: [deployment](04_deployment.ipynb).